# **CDS Project: Part 1**

*Institute of Software Security (E22)*  
*Hamburg University of Technology*  
*SoSe 2023*

## Learning objectives
---

- Use a basic Machine Learning (ML) pipeline with pre-trained models.
- Build your own data loader.
- Load and run a pre-trained ML model.
- Evaluate the performance of an ML model.
- Calculate and interpret performance metrics.

## Materials
---

- Lecture Slides 1, 2, and 3.
- PyTorch Documentation: [Datasets and Data Loaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html) 


## Project Description
---

In this project, you are given an ML model that is pre-trained on a vulnerability dataset. The dataset consists of code samples labeled with True or False flags, depending on the presence and absense of a vulnerability. Your goal is to use the pre-trained model to predict if the code samples in the validation set contain vulnerabilities or not and analyse the results. Please proceed to the below tasks. 

### *Task 1*

Build a data loader for the validation dataset present in the following path: "*data_students/student_dataset.hdf5*". You will be using this dataset to validate the performance of the ML model. The dataset is in HDF5 binary data format. This format is used to store large amount of data. Make sure that you import and familiarise yourself with the right Python libraries to handle HDF5 files. 


In [ ]:
# Open and load the dataset

import h5py
import torch
from torch.utils.data import Dataset, DataLoader

file = h5py.File("data_students/student_dataset.hdf5", "r")

print(file.keys())

<KeysViewHDF5 ['labels', 'source', 'vectors']>


In [ ]:
print(file["vectors"].shape)
print(file["labels"].shape)
print(file["source"].shape)

(1000, 1, 768)
(1000,)
(1000,)


In [ ]:
class VulnerabilityDataset(Dataset):
    """Custom Dataset for loading vulnerability data from HDF5."""
    def __init__(self, file_name):

        self.vectors = file["vectors"][:]
        self.labels  = file["labels"][:]

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, i):
        
        vector = torch.FloatTensor(self.vectors[i].squeeze())   # (768,)
        label  = torch.FloatTensor([int(self.labels[i])])       # (1,)
        return vector, label

# Create dataset and dataloader
dataset =  VulnerabilityDataset(file)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

print(f"\nDataset size : {len(dataset)} samples")
print(f"Num batches  : {len(dataloader)} (batch_size=32)")



Dataset size : 1000 samples
Num batches  : 32 (batch_size=32)


### *Task 2*

Generate a table with 10 random samples from the dataset and show their corresponding labels.

In [ ]:
import pandas as pd
import numpy as np
import random

vectors = file["vectors"]
labels = file["labels"]
rows = []

for i in random.sample(range(len(vectors)),10):
    rows.append({
        "index" : i,
        "vectors" : vectors[i][:10],
        "labels" : labels[i]
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

 index                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

### *Task 3*

Inspect the dataset and answer the following questions:
1.  How many samples are in the dataset?
2. How many positive examples (vulnerability-labeled instances) are in the dataset?
3. What is the vulnerable/non-vulnerable ratio?

In [ ]:
# Inspect and understand the loaded dataset
labels =  [dataset[i][1].item() for i in range(len(dataset))]

total_samples = len(labels)
print(f"Number of samples : {total_samples}")
num_vulnerabilities =  sum(labels)
print(f"Number of postive instances: {num_vulnerabilities}")
num_not_vulnerable = total_samples - num_vulnerabilities

ratio = num_vulnerabilities/num_not_vulnerable
print(f"vulnerable/non-vulnerable ratio : {ratio}")


Number of samples : 1000
Number of postive instances: 283.0
vulnerable/non-vulnerable ratio : 0.3947001394700139


###*Task 4*

Load and run the following pre-trained neural network model called VulnPredictionModel. 

In [28]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")


Using cpu device


In [ ]:
from torch import nn

class VulnPredictModel(nn.Module):
    # intializing the model architecture
    def __init__(self):
      super().__init__()
      self.flatten = nn.Flatten()
      self.linear_stack = nn.Sequential(
         nn.Linear(768, 64),
         nn.ReLU(),
         nn.Linear(64, 64),
         nn.ReLU(),
         nn.Linear(64, 1),
         nn.Sigmoid()
      )

    # forward propagation
    def forward(self, x):
      pred = self.linear_stack(x)
      return pred
      

model = VulnPredictModel().to(device)
print(model)

model.load_state_dict(
    torch.load("model_2023-03-28_20-03.pth", map_location=device)
)

model.eval()
print("\nModel loaded and set to eval mode.") 


VulnPredictModel(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_stack): Sequential(
    (0): Linear(in_features=768, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
    (5): Sigmoid()
  )
)

Model loaded and set to eval mode.


### *Task 5*

Make a prediction on the provided dataset and compute the following values:
- True Positives
- True Negatives
- False Positives
- False Negatives

In [ ]:
# Make the prediction for all the samples in the validation set.
import numpy as np
all_preds  = []   
all_labels = []   

with torch.no_grad():
    for vectors_batch, labels_batch in dataloader:
       
        vectors_batch = vectors_batch.to(device)   
        labels_batch  = labels_batch.to(device)    
        
        outputs = model(vectors_batch)           

        predicted = (outputs >= 0.3).float()      

        all_preds.extend(predicted.cpu().numpy().flatten())
        all_labels.extend(labels_batch.cpu().numpy().flatten())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)


TP = int(np.sum((all_preds == 1) & (all_labels == 1)))

TN = int(np.sum((all_preds == 0) & (all_labels == 0)))

FP = int(np.sum((all_preds == 1) & (all_labels == 0)))

FN = int(np.sum((all_preds == 0) & (all_labels == 1)))


print(f"  True Positives  (TP): {TP}")
print(f"  True Negatives  (TN): {TN}")
print(f"  False Positives (FP): {FP}")
print(f"  False Negatives (FN): {FN}")
print(f"  Total samples       : {TP+TN+FP+FN}")

  True Positives  (TP): 94
  True Negatives  (TN): 704
  False Positives (FP): 13
  False Negatives (FN): 189
  Total samples       : 1000


### *Task 6*

Compute the corresponding performance metrics **manually** (do not use PyTorch's predefined metrics):
- Accuracy
- Precision
- Recall
- F1

In [ ]:
# calculate accuracy
accuracy = (TP + TN) / (TP + TN + FP + FN)
# calculate precision
precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
# calculate recall
recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
# calculate F1-score
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("Performance Metrics")
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1 Score  : {f1:.4f}")

Performance Metrics
  Accuracy  : 0.7980
  Precision : 0.8785
  Recall    : 0.3322
  F1 Score  : 0.4821


### *Task 7*

Based on your performance metrics, answer the following questions:

- Explain the impact of accuracy vs. F1 score.

    Accuracy counts all correct predictions equally, It misleads on imbalanced data. On an imbalanced dataset, where safe samples outnumber vulnerable ones, a model that always predicts safe can score 90% accuracy while missing every real vulnerability. F1 score avoids this trap by combining precision and recall into a single measure, so it only rewards the model when it actually finds vulnerabilities correctly, not just when it gets the majority class right.

- In this particular problem, which metric one should focus more on?

    Recall should be the primary focus. In vulnerability detection, a false negative means a real security flaw goes undetected, which is far more dangerous than a false alarm. F1 score is also important as a secondary metric because it prevents the model from overcorrecting by flagging everything as vulnerable.

- Is there a better metric suitable for the use case of vulnerability prediction? Why?

    Our recall is only 7.1% at threshold=0.5, which means most real vulnerabilities are being assigned a probability below 0.5 by the model. Instead of hardcoding this threshold, AUC-ROC evaluates the model across all possible thresholds and summarises performance as a single score, the AUC value. 

